# Amplitude compensation


## 0. Driver check


In [ ]:
import spcm
from spcm import units

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    print(f"Serial number:    {card.sn()}")
    print(f"Function type:    {card.function_type()}")
    print(f"Max sample value: {card.max_sample_value()}")
    print(
        f"Max sample rate:  {spcm.Clock(card).sample_rate(max=True, return_unit=units.MHz)}"
    )

print("Driver check OK -- card opened, queried, and closed without errors.")

## 1. Setup


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from atommovr.utils.Move import Move
from atommovr.utils.core import PhysicalParams

from awg_controller import (
    AmplitudeCompensation,
    AODSettings,
    AWGEngine,
    AWGEngineConfig,
    CardConfig,
    RFConverter,
)
from awg_controller.awg_control import REFERENCE_FREQUENCY_HZ

CARD_PATH = "/dev/spcm0"
MAX_AMP_V = 1.0  # into 50 ohm
engine = None


def open_engine(cfg, f_lo, f_hi, grid):
    global engine
    if engine is not None:
        engine.close()
        engine = None
    aod = AODSettings(
        f_min_v=f_lo,
        f_max_v=f_hi,
        f_min_h=f_lo,
        f_max_h=f_hi,
        grid_rows=grid,
        grid_cols=grid,
    )
    card = CardConfig(card_path=CARD_PATH, max_amplitude_v=MAX_AMP_V, aod_settings=aod)
    assert card.max_amplitude_v <= 2.0, "exceeds hard safety ceiling"
    engine = AWGEngine(card, cfg)
    fs = engine.open()
    print(
        f"mode={cfg.mode}  fs={fs / 1e6:.1f} MHz  tones={grid}+{grid}  "
        f"Nyquist={fs / 2e6:.0f} MHz  tones {f_lo / 1e6:.1f}..{f_hi / 1e6:.1f} MHz"
    )
    print(f"max round = {engine.max_round_duration_s * 1e3:.1f} ms")
    return fs

## 2. Calibration data

In [ ]:
F_LO, F_HI = 80e6, 110e6

# TODO: replace with measured (frequency, power) calibration data.
CAL_FREQS_HZ = np.linspace(F_LO, F_HI, 21)
rng = np.random.default_rng(0)
CAL_POWERS = 1.0 - 0.25 * np.exp(-((CAL_FREQS_HZ - 95e6) ** 2) / (2 * 8e6**2))
CAL_POWERS += rng.normal(scale=0.01, size=CAL_POWERS.shape)

plt.figure(figsize=(6, 3.5))
plt.plot(CAL_FREQS_HZ / 1e6, CAL_POWERS, "o", label="synthesized data")
plt.xlabel("frequency (MHz)")
plt.ylabel("relative power")
plt.title("Placeholder calibration data")
plt.legend()
plt.tight_layout()

## 3. Fit compensation curves

In [ ]:
linear_comp = AmplitudeCompensation.fit_linear(CAL_FREQS_HZ, CAL_POWERS)
gaussian_comp = AmplitudeCompensation.fit_gaussian(CAL_FREQS_HZ, CAL_POWERS)

f_plot = np.linspace(F_LO, F_HI, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(CAL_FREQS_HZ / 1e6, CAL_POWERS, "o", label="data")
plt.plot(f_plot / 1e6, [linear_comp(f) for f in f_plot], label="linear fit")
plt.plot(f_plot / 1e6, [gaussian_comp(f) for f in f_plot], label="gaussian fit")
plt.axvline(REFERENCE_FREQUENCY_HZ / 1e6, color="gray", ls="--", lw=1, label="reference (100 MHz)")
plt.xlabel("frequency (MHz)")
plt.ylabel("amplitude ratio")
plt.legend()
plt.tight_layout()

print(f"linear:   a={linear_comp.a:.4f}  b={linear_comp.b:.3e} /Hz")
print(
    f"gaussian: a={gaussian_comp.a:.4f}  b={gaussian_comp.b:.4f}  "
    f"f0={gaussian_comp.f0_hz / 1e6:.2f} MHz  sigma={gaussian_comp.sigma_hz / 1e6:.2f} MHz"
)

## 4. Compensated round

In [ ]:
GRID = 4
COMP = gaussian_comp  # pick which fit drives the hardware round
REFERENCE_AMP_PCT = 0.8 * 40.0 / GRID  # headroom below the flat 40%/tone share

aod = AODSettings(
    f_min_v=F_LO,
    f_max_v=F_HI,
    f_min_h=F_LO,
    f_max_h=F_HI,
    grid_rows=GRID,
    grid_cols=GRID,
)
converter = RFConverter(
    aod,
    PhysicalParams(),
    amplitude_compensation=COMP,
    reference_amplitude_pct=REFERENCE_AMP_PCT,
)


def one_step_dest(i, n):
    return i + 1 if i + 1 < n else i - 1


def compensated_one_step_round(step_s):
    batches = [converter.holding_config()]  # settle
    for i in range(GRID):
        batches.append(converter.convert_moves([Move(i, 0, one_step_dest(i, GRID), 0)]))
    for j in range(GRID):
        batches.append(converter.convert_moves([Move(0, j, 0, one_step_dest(j, GRID))]))
    batches.append(converter.holding_config())  # park
    for b in batches:
        b.travel_duration_s = step_s
        for r in b.ramps:
            r.duration_s = step_s if r.f_start != r.f_end else 0.0
    return batches


home = converter.holding_config()
row_amps = [r.amplitude_pct for r in home.ramps if r.channel == 0]
print(f"{GRID}x{GRID}  reference={REFERENCE_AMP_PCT:.2f}%/tone")
print("row amplitudes: " + ", ".join(f"{a:.2f}%" for a in row_amps))
print(f"row channel total: {sum(row_amps):.2f}% (budget 40%)")

## 5. Play on hardware


In [ ]:
fs = open_engine(AWGEngineConfig(mode="stream"), F_LO, F_HI, grid=GRID)
print(f"ring = {engine.look_ahead_s * 1e3:.1f} ms of look-ahead")

batches = compensated_one_step_round(step_s=1)
n_moving = sum(1 for b in batches if any(r.f_start != r.f_end for r in b.ramps))
print(f"{len(batches)} batches ({n_moving} moving), {len(batches[0].ramps)} ramps each")

engine.load_round(batches)
print(f"loaded {engine.total_travel_duration_s:.2f} s of waveform")

print("Playing -- amplitude tracks the compensation curve across each sweep, not just its endpoints.")
engine.play()
assert engine.last_error is None, engine.last_error

time.sleep(engine.total_travel_duration_s + 1.0)
assert engine.last_error is None, engine.last_error
print("Compensated one-step row/col sweep complete; grid parked at its start sites until close().")

## 6. Cleanup


In [ ]:
if engine is not None:
    engine.close()
    engine = None
    print("Engine closed.")